<a href="https://colab.research.google.com/github/thansihatech/CSA0816-PYTHON/blob/main/ASSIGNMENT_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

import re
import time

# Normalize phone numbers
def normalize_phone(phone):
    digits = re.sub(r"\D", "", phone or "")

    if len(digits) == 12 and digits.startswith("91"):
        digits = digits[2:]
    elif len(digits) == 11 and digits.startswith("0"):
        digits = digits[1:]

    return digits


# Normalize names
def normalize_name(name):
    return " ".join((name or "").lower().split())


# Normalize emails
def normalize_email(email):
    return (email or "").strip().lower()


# Merge contacts using a dictionary
def merge_contacts(contacts):
    phone_index = {}
    groups = []

    for contact in contacts:
        name = normalize_name(contact.get("name", ""))
        phone = normalize_phone(contact.get("phone", ""))
        email = normalize_email(contact.get("email", ""))

        # Find existing groups using matching phone numbers
        matched = {
            id(phone_index[p]): phone_index[p]
            for p in [phone]
            if p and p in phone_index
        }

        # Create a new group if no match exists
        if not matched:
            group = {
                "names": set(),
                "phones": set(),
                "emails": set()
            }
            groups.append(group)
        else:
            group = next(iter(matched.values()))

            # Merge other matching groups
            for other in matched.values():
                if other is not group:
                    group["names"].update(other["names"])
                    group["phones"].update(other["phones"])
                    group["emails"].update(other["emails"])

                    for p in other["phones"]:
                        phone_index[p] = group

                    groups.remove(other)

        # Add contact details
        if name:
            group["names"].add(name)
        if phone:
            group["phones"].add(phone)
            phone_index[phone] = group
        if email:
            group["emails"].add(email)

    return groups


# Sample contacts
contacts = [
    {"name": "Dr. Ravi", "phone": "+91 98400 12345",
     "email": "ravi@gmail.com"},

    {"name": "ravi kumar", "phone": "9840012345",
     "email": "ravi@gmail.com"},

    {"name": "Priya", "phone": "9876543210",
     "email": "priya@gmail.com"},

    {"name": "Ravi Kumar", "phone": "9840012345",
     "email": "ravi.work@gmail.com"},

    {"name": "Arun", "phone": "",
     "email": "arun@gmail.com"}
]

# Display merged contacts
result = merge_contacts(contacts)

print("MERGED CONTACTS")

for contact in result:
    print("\nNames:", contact["names"])
    print("Phone Numbers:", contact["phones"])
    print("Emails:", contact["emails"])


# Verify performance for 1,000 and 100,000 contacts
for size in [1000, 100000]:
    test_data = [
        {
            "name": f"Person {i}",
            "phone": f"90000{i:06d}",
            "email": f"person{i}@gmail.com"
        }
        for i in range(size)
    ]

    start = time.perf_counter()
    merged = merge_contacts(test_data)
    end = time.perf_counter()

    print("\nContacts:", size)
    print("Merged Groups:", len(merged))
    print("Execution Time:", round(end - start, 6), "seconds")


MERGED CONTACTS

Names: {'ravi kumar', 'dr. ravi'}
Phone Numbers: {'9840012345'}
Emails: {'ravi@gmail.com', 'ravi.work@gmail.com'}

Names: {'priya'}
Phone Numbers: {'9876543210'}
Emails: {'priya@gmail.com'}

Names: {'arun'}
Phone Numbers: set()
Emails: {'arun@gmail.com'}

Contacts: 1000
Merged Groups: 1000
Execution Time: 0.002635 seconds

Contacts: 100000
Merged Groups: 100000
Execution Time: 0.53987 seconds
